# TP 1.2 — Données éducation via les API (data.gouv.fr & data.education.gouv.fr)

**Objectif** : interroger le **catalogue** et les **jeux éducation** en HTTP depuis un notebook Python, sans passer uniquement par un téléchargement manuel.

**Durée indicative** : 1 h 30

**Prérequis** : Python 3.10+, `pandas`, connexion Internet. Lancer Jupyter **depuis la racine du dépôt** (dossier qui contient `donnees/`).

**Références** : `Docs/sources_data.md` — [documentation API data.gouv.fr](https://www.data.gouv.fr/fr/dataservices/api/) — [portail data.education.gouv.fr](https://data.education.gouv.fr/)

---

### MCP Cursor vs API dans le notebook

| Contexte | Rôle |
|----------|------|
| **MCP `datagouv`** (dans Cursor) | L’assistant explore le catalogue pour vous (`search_datasets`, `query_resource_data`, …). |
| **API REST** (ce TP) | Votre **code** appelle les mêmes portails : reproductible, scriptable, intégrable dans un pipeline. |

Les deux accèdent aux **mêmes jeux publics** ; seul le mode d’appel change.

## 0. Configuration

In [ ]:
from pathlib import Path
import json
import urllib.parse
import urllib.request

import pandas as pd


def resolve_data_dir() -> Path:
    """Racine du dépôt = dossier qui contient `donnees/`."""
    cwd = Path.cwd()
    if (cwd / "donnees").is_dir():
        return cwd
    p = cwd
    for _ in range(4):
        if (p / "donnees").is_dir():
            return p
        p = p.parent
    return cwd


ROOT = resolve_data_dir()
DATA = ROOT / "donnees"
DATA.mkdir(exist_ok=True)

# Identifiants stables des jeux utilisés dans la formation
DATASET_ANNUAIRE_ID = "5889d03fa3a72974cbf0d5b1"
ODS_DATASET_ANNUAIRE = "fr-en-annuaire-education"

API_DATAGOUV_V1 = "https://www.data.gouv.fr/api/1"
API_DATAGOUV_V2 = "https://www.data.gouv.fr/api/2"
API_EDUCATION = "https://data.education.gouv.fr/api/explore/v2.1/catalog"

USER_AGENT = "TP-BigData-OpenData-Formation/1.0 (usage pedagogique)"
print("Données locales :", DATA.resolve())

In [ ]:
def http_get_json(url: str, params: dict | None = None, timeout: int = 60) -> dict | list:
    """GET JSON avec la bibliothèque standard (pas de clé API requise pour l'open data)."""
    if params:
        url = f"{url}?{urllib.parse.urlencode(params)}"
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def records_to_dataframe(payload: dict) -> pd.DataFrame:
    """Convertit la réponse `records` d'OpenDataSoft en DataFrame."""
    return pd.DataFrame(payload.get("results", []))

## 1. API catalogue **data.gouv.fr**

Découverte de jeux par mot-clé, lecture des métadonnées et des ressources (fichiers CSV, JSON, …).

### 1.1 Rechercher des jeux « éducation »

In [ ]:
# API v1 : recherche simple par requête textuelle
search = http_get_json(
    f"{API_DATAGOUV_V1}/datasets/",
    {"q": "annuaire education", "page_size": 5},
)
print("Nombre de résultats (approx.) :", search.get("total"))
for ds in search.get("data", [])[:5]:
    org = (ds.get("organization") or {}).get("name", "?")
    print(f"- {ds['title']} | org={org} | id={ds['id']}")

**À faire** : modifier la requête `q` pour trouver un jeu sur les **IVAL** ou les **effectifs**. Noter l’`id` du jeu retenu.

In [ ]:
# TODO : votre recherche
MA_RECHERCHE = "..."  # ex. "IVAL lycée"

resultats = http_get_json(
    f"{API_DATAGOUV_V1}/datasets/",
    {"q": MA_RECHERCHE, "page_size": 10},
)
pd.DataFrame(
    [
        {"titre": d["title"], "id": d["id"], "page": d.get("page")}
        for d in resultats.get("data", [])
    ]
)

### 1.2 Métadonnées d’un jeu (fiche détaillée)

In [ ]:
fiche = http_get_json(f"{API_DATAGOUV_V1}/datasets/{DATASET_ANNUAIRE_ID}/")
print("Titre :", fiche["title"])
print("Licence :", fiche.get("license"))
print("Dernière MAJ :", fiche.get("last_update"))
print("Description (extrait) :", (fiche.get("description") or "")[:200], "…")

### 1.3 Lister les ressources (fichiers) d’un jeu — API v2

In [ ]:
resources = http_get_json(
    f"{API_DATAGOUV_V2}/datasets/{DATASET_ANNUAIRE_ID}/resources/",
    {"page_size": 20},
)
df_res = pd.DataFrame(
    [
        {
            "titre": r.get("title"),
            "format": r.get("format"),
            "taille": r.get("filesize"),
            "url": r.get("url"),
        }
        for r in resources.get("data", [])
    ]
)
df_res

**Question** : quelle URL permet de récupérer l’export CSV « à jour » hébergé sur `data.education.gouv.fr` ? (indice : format `csv`, titre contenant `fr-en-annuaire`)

## 2. API **data.education.gouv.fr** (OpenDataSoft Explore v2.1)

Interroger les enregistrements **ligne par ligne** (filtres, pagination) — utile pour un échantillon ou une extraction ciblée.

### 2.1 Aperçu : quelques établissements

In [ ]:
base = f"{API_EDUCATION}/datasets/{ODS_DATASET_ANNUAIRE}/records"
params = {
    "limit": 5,
    "select": "identifiant_de_l_etablissement,nom_etablissement,type_etablissement,nom_commune,code_departement",
}
apercu = http_get_json(base, params)
print("Total dans le jeu :", apercu.get("total_count"))
records_to_dataframe(apercu)

### 2.2 Filtrer avec `where` (langage ODSQL)

Exemple : lycées publics du département de Paris (`code_departement` = `075`).

In [ ]:
params_filtre = {
    "limit": 10,
    "select": "nom_etablissement,nom_commune,statut_public_prive,libelle_academie",
    "where": 'code_departement="075" AND type_etablissement="Lycée" AND statut_public_prive="Public"',
}
lycees_paris = http_get_json(base, params_filtre)
print("Nombre total correspondant :", lycees_paris.get("total_count"))
records_to_dataframe(lycees_paris)

**À faire** : adapter le filtre `where` pour lister les **collèges** d’**un département de votre choix** (code sur 3 chiffres, ex. `033` pour Gironde). Afficher au moins 5 lignes.

In [ ]:
# TODO : remplacer CODE_DEPARTEMENT et vérifier le résultat
CODE_DEPARTEMENT = "033"  # à modifier

params_exo = {
    "limit": 10,
    "select": "nom_etablissement,nom_commune,type_etablissement",
    "where": f'code_departement="{CODE_DEPARTEMENT}" AND type_etablissement="Collège"',
}
# votre code ici


### 2.3 Pagination (`limit` / `offset`)

Pour parcourir un gros jeu sans tout charger d’un coup.

In [ ]:
def fetch_all_records(base_url: str, page_size: int = 100, max_pages: int = 3, **extra_params) -> pd.DataFrame:
    """Récupère plusieurs pages (démo : max_pages limité volontairement)."""
    frames = []
    for page in range(max_pages):
        params = {"limit": page_size, "offset": page * page_size, **extra_params}
        payload = http_get_json(base_url, params)
        batch = records_to_dataframe(payload)
        if batch.empty:
            break
        frames.append(batch)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


echantillon = fetch_all_records(
    base,
    page_size=50,
    max_pages=2,
    select="type_etablissement,code_region",
)
print(len(echantillon), "lignes")
echantillon["type_etablissement"].value_counts().head()

### 2.4 Télécharger un export CSV via l’API (alternative au navigateur)

Même URL que dans `Docs/sources_data.md` — pratique pour automatiser la mise à jour du dossier `donnees/`.

In [ ]:
EXPORT_CSV_URL = (
    f"{API_EDUCATION}/datasets/{ODS_DATASET_ANNUAIRE}/exports/csv"
)
dest = DATA / "fr-en-annuaire-education-api.csv"

# Décommenter pour télécharger (~40 Mo) — peut prendre 1 à 3 minutes
# req = urllib.request.Request(EXPORT_CSV_URL, headers={"User-Agent": USER_AGENT})
# with urllib.request.urlopen(req, timeout=300) as resp, dest.open("wb") as f:
#     f.write(resp.read())
# print("Enregistré :", dest, "|", dest.stat().st_size // 1_000_000, "Mo")

if dest.exists():
    print("Fichier déjà présent :", dest)
else:
    print("Export non téléchargé. Utilisez l'URL :", EXPORT_CSV_URL)

## 3. Comparer API et fichier local

Si vous avez déjà `donnees/fr-en-annuaire-education.csv` (TP 1.1), comparez le volume avec `total_count` de l’API.

In [ ]:
csv_local = DATA / "fr-en-annuaire-education.csv"
meta = http_get_json(f"{API_EDUCATION}/datasets/{ODS_DATASET_ANNUAIRE}")

print("Lignes côté API (total_count) :", meta.get("dataset_id") and http_get_json(base, {"limit": 1}).get("total_count"))

if csv_local.exists():
    # Comptage rapide des lignes (sans tout charger en RAM si besoin : wc -l en shell)
    n_local = sum(1 for _ in open(csv_local, encoding="utf-8", errors="replace")) - 1
    print("Lignes fichier local :", n_local)
else:
    print("Pas de CSV local — téléchargement manuel ou cellule 2.4")

## 4. Synthèse — à rendre (markdown ou oral)

1. **Trois endpoints** utilisés dans ce TP et leur rôle (catalogue vs enregistrements vs export).
2. **Quand préférer** l’API `records` vs un **export CSV complet** ? (volume, fréquence de MAJ, reproductibilité.)
3. **Équivalent MCP** : pour chaque étape du TP, indiquer quel outil MCP (`search_datasets`, `get_dataset_info`, …) ferait la même chose dans Cursor.
4. **Limites** : débit, disponibilité réseau, colonnes renommées entre versions — comment les gérer en projet ?

## 5. Aller plus loin (optionnel)

- API v2 recherche : `GET https://www.data.gouv.fr/api/2/datasets/search/?q=education`
- Jeu **IVAL GT** : dataset ODS `fr-en-indicateurs-de-resultat-des-lycees-gt_v2`
- [API calendrier scolaire](https://www.data.gouv.fr/dataservices/api-calendrier-scolaire/) sur data.education.gouv.fr
- Bibliothèque `requests` (plus lisible que `urllib`) — voir le notebook corrigé `TP_corrige/jour1/TP_1_2_API_data_gouv_education.ipynb`